# Data Science Salary Dashboard - Data Preprocessing

This notebook loads all 5 datasets, normalizes schemas, merges them, and saves
a single cleaned DataFrame to `data/processed/salary_merged.csv`.

In [17]:
import pandas as pd
import numpy as np
import os
import re

os.chdir(r"D:\Desktop\Data Science Salary Dashboard")
RAW = "data/raw"
OUT = "data/processed"
os.makedirs(OUT, exist_ok=True)

## 1. Load All Datasets

In [18]:
df_orig1 = pd.read_csv(f"{RAW}/Data Science Salary 2021 to 2023.csv")
df_orig2 = pd.read_csv(f"{RAW}/data_science_salaries.csv")
df_orig3 = pd.read_csv(f"{RAW}/salaries.csv")
df_tech  = pd.read_csv(f"{RAW}/global_ai_tech_salaries_2020_2025.csv")
df_ai    = pd.read_csv(f"{RAW}/global_ai_jobs.csv")

print(f"orig1: {df_orig1.shape}")
print(f"orig2: {df_orig2.shape}")
print(f"orig3: {df_orig3.shape}")
print(f"tech:  {df_tech.shape}")
print(f"ai:    {df_ai.shape}")

orig1: (3761, 9)
orig2: (6599, 11)
orig3: (151445, 11)
tech:  (12000, 22)
ai:    (90000, 35)


## 2. Define Unified Schema

Target columns for the merged dataset:

| Column | Type | Source |
|--------|------|--------|
| `work_year` | int | all |
| `job_title` | str | all (mapped from `job_role` in ai_jobs) |
| `experience_level` | str | all (normalized) |
| `experience_years` | float | ai_jobs, tech (estimated for others) |
| `employment_type` | str | all (normalized) |
| `salary_in_usd` | int | all (mapped from `base_salary_usd` / `salary_usd`) |
| `company_location` | str | all |
| `company_size` | str | all (normalized) |
| `education_level` | str | tech, ai_jobs |
| `work_setting` | str | tech, ai_jobs, orig2 |
| `industry` | str | tech, ai_jobs |
| `primary_skills` | str | tech |
| `remote_ratio` | int | orig3 |
| `ai_specialization` | str | ai_jobs |
| `cost_of_living_index` | float | ai_jobs |
| `job_security_score` | float | ai_jobs |
| `work_life_balance_score` | float | ai_jobs |
| `ai_adoption_score` | float | ai_jobs |
| `bonus_usd` | float | ai_jobs, tech |

## 3. Normalize Experience Level

In [19]:
EXP_MAP = {
    # orig1 & orig3 abbreviations
    "EN": "Entry", "MI": "Mid", "SE": "Senior", "EX": "Executive",
    # orig2 full names
    "Entry-level": "Entry", "Mid-level": "Mid",
    "Senior-level": "Senior", "Executive-level": "Executive",
    # tech_salaries
    "Entry (0-2 yrs)": "Entry", "Mid (3-5 yrs)": "Mid",
    "Senior (6-10 yrs)": "Senior", "Lead (11-15 yrs)": "Lead",
    "Executive (16+ yrs)": "Executive",
    # ai_jobs
    "Entry": "Entry", "Mid": "Mid", "Senior": "Senior", "Lead": "Lead",
}

def norm_experience(val):
    if pd.isna(val):
        return val
    return EXP_MAP.get(str(val).strip(), str(val).strip())

## 4. Normalize Employment Type

In [20]:
EMP_MAP = {
    "FT": "Full-time", "PT": "Part-time", "CT": "Contract", "FL": "Freelance",
    "Full Time": "Full-time", "Part Time": "Part-time",
    "Full-time": "Full-time", "Part-time": "Part-time",
    "Contract": "Contract", "Freelance": "Freelance",
}

def norm_employment(val):
    if pd.isna(val):
        return val
    return EMP_MAP.get(str(val).strip(), str(val).strip())

## 5. Normalize Company Size

In [21]:
SIZE_MAP = {
    "S": "Small", "M": "Medium", "L": "Large",
    "Small": "Small", "Medium": "Medium", "Large": "Large",
    "Startup (1-50)": "Small", "Small (51-200)": "Small",
    "Medium (201-1000)": "Medium", "Large (1001-10000)": "Large",
    "Enterprise (10000+)": "Enterprise",
    "Startup": "Small", "Enterprise": "Enterprise",
}

def norm_company_size(val):
    if pd.isna(val):
        return val
    return SIZE_MAP.get(str(val).strip(), str(val).strip())

## 6. Normalize Work Setting

In [22]:
WORK_MAP = {
    "On-site": "Onsite", "Onsite": "Onsite", "Remote": "Remote",
    "Hybrid": "Hybrid",
}

def norm_work_setting(val):
    if pd.isna(val):
        return val
    return WORK_MAP.get(str(val).strip(), str(val).strip())

## 7. Process Original Datasets (orig1, orig2, orig3)

In [23]:
def process_originals():
    frames = []

    # --- orig1: Data Science Salary 2021 to 2023 ---
    d = df_orig1.copy()
    d = d.rename(columns={"salary": "salary_local"})
    d["work_year"] = d["work_year"].astype(int)
    d["experience_level"] = d["experience_level"].apply(norm_experience)
    d["employment_type"] = d["employment_type"].apply(norm_employment)
    d["company_size"] = d["company_size"].apply(norm_company_size)
    d["source"] = "orig1"
    frames.append(d)
    print(f"orig1 processed: {d.shape}")

    # --- orig2: data_science_salaries ---
    d = df_orig2.copy()
    d = d.rename(columns={"salary": "salary_local", "work_models": "work_setting"})
    d["work_year"] = d["work_year"].astype(int)
    d["experience_level"] = d["experience_level"].apply(norm_experience)
    d["employment_type"] = d["employment_type"].apply(norm_employment)
    d["company_size"] = d["company_size"].apply(norm_company_size)
    d["work_setting"] = d["work_setting"].apply(norm_work_setting)
    d["source"] = "orig2"
    frames.append(d)
    print(f"orig2 processed: {d.shape}")

    # --- orig3: salaries.csv (151K rows) ---
    d = df_orig3.copy()
    d = d.rename(columns={"salary": "salary_local"})
    d["work_year"] = d["work_year"].astype(int)
    d["experience_level"] = d["experience_level"].apply(norm_experience)
    d["employment_type"] = d["employment_type"].apply(norm_employment)
    d["company_size"] = d["company_size"].apply(norm_company_size)
    d["source"] = "orig3"
    frames.append(d)
    print(f"orig3 processed: {d.shape}")

    # Drop exact duplicates across the original datasets
    combined = pd.concat(frames, ignore_index=True, sort=False)
    before = len(combined)
    combined.drop_duplicates(
        subset=["work_year", "experience_level", "employment_type",
                "job_title", "salary_in_usd", "company_location", "company_size"],
        inplace=True
    )
    print(f"Originals combined: {before} -> {len(combined)} rows (removed {before - len(combined)} dupes)")
    return combined

df_originals = process_originals()

orig1 processed: (3761, 10)
orig2 processed: (6599, 12)
orig3 processed: (151445, 12)
Originals combined: 161805 -> 72745 rows (removed 89060 dupes)


## 8. Process Tech Salaries Dataset

In [24]:
def process_tech():
    d = df_tech.copy()
    d = d.rename(columns={
        "survey_year": "work_year",
        "base_salary_usd": "salary_in_usd",
        "country": "company_location",
    })
    d["work_year"] = d["work_year"].astype(int)
    d["experience_level"] = d["experience_level"].apply(norm_experience)
    d["employment_type"] = d["employment_type"].apply(norm_employment)
    d["company_size"] = d["company_size"].apply(norm_company_size)
    d["work_setting"] = d["work_setting"].apply(norm_work_setting)
    d["source"] = "tech"
    print(f"tech processed: {d.shape}")
    return d

df_tech_clean = process_tech()
print(df_tech_clean[["work_year", "job_title", "experience_level", "education_level",
                      "work_setting", "industry", "salary_in_usd", "primary_skills"]].head())

tech processed: (12000, 23)
   work_year               job_title experience_level education_level  \
0       2023          Data Scientist              Mid        Master's   
1       2021  Generative AI Engineer              Mid             PhD   
2       2022       Backend Developer           Senior             PhD   
3       2022            Data Analyst             Lead             PhD   
4       2021      Frontend Developer             Lead      Bachelor's   

  work_setting             industry  salary_in_usd  \
0       Hybrid           Technology         112600   
1       Hybrid           Technology         198800   
2       Remote  Healthcare & Pharma          52800   
3       Remote  Healthcare & Pharma          50700   
4       Hybrid           Technology         216500   

                          primary_skills  
0                         JavaScript; Go  
1  Scala; JavaScript; LangChain; PyTorch  
2      Tableau; JavaScript; Airflow; SQL  
3                        Power BI; E

## 9. Process AI Jobs Dataset

In [25]:
def process_ai_jobs():
    d = df_ai.copy()
    d = d.rename(columns={
        "job_role": "job_title",
        "year": "work_year",
        "salary_usd": "salary_in_usd",
        "work_mode": "work_setting",
        "education_required": "education_level",
        "country": "company_location",
    })
    d["work_year"] = d["work_year"].astype(int)
    d["experience_level"] = d["experience_level"].apply(norm_experience)
    d["company_size"] = d["company_size"].apply(norm_company_size)
    d["work_setting"] = d["work_setting"].apply(norm_work_setting)
    d["education_level"] = d["education_level"].map({
        "Bootcamp": "Other", "Diploma": "Other",
        "Bachelor": "Bachelor's", "Master": "Master's", "PhD": "PhD",
    })
    # ai_jobs has no employment_type column, default to Full-time
    d["employment_type"] = "Full-time"
    d["source"] = "ai_jobs"
    print(f"ai_jobs processed: {d.shape}")
    return d

df_ai_clean = process_ai_jobs()
print(df_ai_clean[["work_year", "job_title", "experience_level", "experience_years",
                    "education_level", "work_setting", "industry", "salary_in_usd"]].head())

ai_jobs processed: (90000, 37)
   work_year                  job_title experience_level  experience_years  \
0       2020  Machine Learning Engineer            Entry                 0   
1       2021                AI Engineer            Entry                 1   
2       2020         Research Scientist            Entry                 0   
3       2026       Software Engineer AI           Senior                 6   
4       2021  Machine Learning Engineer            Entry                 0   

  education_level work_setting    industry  salary_in_usd  
0        Master's       Remote  Automotive          66465  
1           Other       Onsite      Retail          75507  
2             PhD       Remote  Healthcare          41660  
3           Other       Onsite        Tech          43268  
4        Master's       Hybrid      Retail          69119  


## 10. Merge Everything Into Unified Schema

In [26]:
# Unified columns
UNIFIED_COLS = [
    "work_year", "job_title", "experience_level", "experience_years",
    "employment_type", "salary_in_usd", "company_location", "company_size",
    "education_level", "work_setting", "industry", "primary_skills",
    "remote_ratio", "ai_specialization", "cost_of_living_index",
    "job_security_score", "work_life_balance_score", "ai_adoption_score",
    "bonus_usd", "job_category", "ai_adoption_level", "gender",
    "job_satisfaction", "years_at_company", "source",
]

def safe_select(df, cols):
    """Select columns that exist, fill missing with NaN."""
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df[cols]

part1 = safe_select(df_originals, UNIFIED_COLS)
part2 = safe_select(df_tech_clean, UNIFIED_COLS)
part3 = safe_select(df_ai_clean, UNIFIED_COLS)

df_all = pd.concat([part1, part2, part3], ignore_index=True)
print(f"Merged: {df_all.shape}")
print(f"\nSource distribution:")
print(df_all["source"].value_counts())

Merged: (174745, 25)

Source distribution:
source
ai_jobs    90000
orig3      64247
tech       12000
orig2       6092
orig1       2406
Name: count, dtype: int64


## 11. Drop Duplicates & Basic Cleaning

In [27]:
# Drop rows with no salary
df_all = df_all.dropna(subset=["salary_in_usd"])

# Remove salary outliers (keep within IQR * 3)
Q1 = df_all["salary_in_usd"].quantile(0.01)
Q3 = df_all["salary_in_usd"].quantile(0.99)
IQR = Q3 - Q1
lower = max(0, Q1 - 1.5 * IQR)
upper = Q3 + 1.5 * IQR
before = len(df_all)
df_all = df_all[(df_all["salary_in_usd"] >= lower) & (df_all["salary_in_usd"] <= upper)]
print(f"Salary outlier filter: {before} -> {len(df_all)} rows")

# Drop exact duplicates
before = len(df_all)
df_all.drop_duplicates(inplace=True)
print(f"Dedup: {before} -> {len(df_all)} rows")

print(f"\nFinal shape: {df_all.shape}")
print(f"\nNull counts:")
print(df_all.isnull().sum())

Salary outlier filter: 174745 -> 174745 rows


Dedup: 174745 -> 174745 rows

Final shape: (174745, 25)

Null counts:
work_year                       0
job_title                       0
experience_level                0
experience_years            84745
employment_type                 0
salary_in_usd                   0
company_location                0
company_size                    0
education_level             72745
work_setting                66653
industry                    72745
primary_skills             162745
remote_ratio               110498
ai_specialization           84745
cost_of_living_index        84745
job_security_score          84745
work_life_balance_score     84745
ai_adoption_score           84745
bonus_usd                   84745
job_category               162745
ai_adoption_level          163775
gender                     162745
job_satisfaction           162745
years_at_company           162745
source                          0
dtype: int64


## 12. Feature Engineering: Extract Top Skills

In [28]:
# Extract top skills from primary_skills (semicolon-separated)
all_skills = []
for val in df_all["primary_skills"].dropna():
    skills = [s.strip() for s in str(val).split(";")]
    all_skills.extend(skills)

skill_counts = pd.Series(all_skills).value_counts()
print("Top 20 skills:")
print(skill_counts.head(20))

# Create binary columns for top 15 skills
TOP_SKILLS = skill_counts.head(15).index.tolist()

def has_skill(row, skill):
    if pd.isna(row):
        return 0
    return 1 if skill in str(row) else 0

for skill in TOP_SKILLS:
    col_name = f"skill_{skill.replace(' ', '_').replace('+', '_').replace('.', '').lower()}"
    df_all[col_name] = df_all["primary_skills"].apply(lambda x: has_skill(x, skill))

print(f"\nAdded {len(TOP_SKILLS)} skill columns")
print(df_all[["primary_skills"] + [c for c in df_all.columns if c.startswith("skill_")]].head())

Top 20 skills:
JavaScript      1618
Rust            1618
Kubernetes      1609
SQL             1606
AWS             1598
Hugging Face    1593
Scala           1591
C++             1582
Airflow         1578
PyTorch         1577
Hadoop          1565
Excel           1562
Kafka           1554
Tableau         1553
R               1553
Power BI        1552
Azure           1537
OpenCV          1528
Go              1528
Docker          1522
Name: count, dtype: int64

Added 15 skill columns
  primary_skills  skill_javascript  skill_rust  skill_kubernetes  skill_sql  \
0            NaN                 0           0                 0          0   
1            NaN                 0           0                 0          0   
2            NaN                 0           0                 0          0   
3            NaN                 0           0                 0          0   
4            NaN                 0           0                 0          0   

   skill_aws  skill_hugging_face  skill_

## 13. Feature Engineering: Skills Count

In [29]:
def count_skills(val):
    if pd.isna(val):
        return 0
    return len([s.strip() for s in str(val).split(";") if s.strip()])

df_all["skill_count"] = df_all["primary_skills"].apply(count_skills)
print("Skill count distribution:")
print(df_all["skill_count"].describe())

Skill count distribution:
count    174745.000000
mean          0.240127
std           0.931945
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           5.000000
Name: skill_count, dtype: float64


## 14. Split Into Two Datasets

In [30]:
# === DATASET A: Original features (all rows) ===
# Keep only the 6 original columns + salary, available across all sources
ORIG_COLS = ["work_year", "job_title", "experience_level", "employment_type",
             "salary_in_usd", "company_location", "company_size"]
df_original = df_all[ORIG_COLS].dropna().copy()

# Remove exact duplicates
before = len(df_original)
df_original.drop_duplicates(inplace=True)
print(f"Dataset A (original features): {df_original.shape} (removed {before - len(df_original)} dupes)")

# === DATASET B: Enriched features (rows with education + industry) ===
before = len(df_all)
df_enriched = df_all[df_all["education_level"] != "Unknown"].copy()
df_enriched = df_enriched[df_enriched["industry"] != "Unknown"]
df_enriched = df_enriched.dropna(subset=["education_level", "industry"])
print(f"Dataset B (enriched features): {df_enriched.shape} (dropped {before - len(df_enriched)} rows)")

# Fill remaining NAs in enriched dataset
cat_fill = ["work_setting", "ai_specialization", "job_category",
            "ai_adoption_level", "gender"]
for col in cat_fill:
    df_enriched[col] = df_enriched[col].fillna("Unknown")

num_fill = ["experience_years", "cost_of_living_index", "job_security_score",
            "work_life_balance_score", "ai_adoption_score", "bonus_usd",
            "remote_ratio", "job_satisfaction", "years_at_company"]
for col in num_fill:
    if col in df_enriched.columns:
        df_enriched[col] = df_enriched[col].fillna(df_enriched[col].median())

exp_years_map = {"Entry": 1, "Mid": 4, "Senior": 8, "Lead": 12, "Executive": 17}
mask = df_enriched["experience_years"].isna()
df_enriched.loc[mask, "experience_years"] = df_enriched.loc[mask, "experience_level"].map(exp_years_map)
df_enriched["experience_years"] = df_enriched["experience_years"].fillna(5)

print(f"\nDataset A nulls: {df_original.isnull().sum().sum()}")
print(f"Dataset B nulls: {df_enriched.isnull().sum().sum()}")

Dataset A (original features): (173453, 7) (removed 1292 dupes)
Dataset B (enriched features): (102000, 41) (dropped 72745 rows)

Dataset A nulls: 0
Dataset B nulls: 192000


## 15. Save All Datasets

In [31]:
# Save Dataset A: original features only (all sources)
path_a = f"{OUT}/salary_original_only.csv"
df_original.to_csv(path_a, index=False)
print(f"Saved: {path_a} ({df_original.shape[0]} rows, {df_original.shape[1]} columns)")

# Save Dataset B: enriched features (ai_jobs + tech only)
path_b = f"{OUT}/salary_model_ready.csv"
df_enriched.to_csv(path_b, index=False)
print(f"Saved: {path_b} ({df_enriched.shape[0]} rows, {df_enriched.shape[1]} columns)")

# Save full merged (everything)
path_full = f"{OUT}/salary_merged.csv"
df_all.to_csv(path_full, index=False)
print(f"Saved: {path_full} ({df_all.shape[0]} rows, {df_all.shape[1]} columns)")

Saved: data/processed/salary_original_only.csv (173453 rows, 7 columns)
Saved: data/processed/salary_model_ready.csv (102000 rows, 41 columns)
Saved: data/processed/salary_merged.csv (174745 rows, 41 columns)


## 16. Summary Statistics

In [32]:
print("=" * 60)
print("DATASET A: Original Features (all sources)")
print("=" * 60)
print(f"Rows: {len(df_original):,}")
print(f"Columns: {list(df_original.columns)}")
print(f"Salary range: ${df_original['salary_in_usd'].min():,.0f} - ${df_original['salary_in_usd'].max():,.0f}")
print(f"Mean salary: ${df_original['salary_in_usd'].mean():,.0f}")
print(f"Experience levels: {sorted(df_original['experience_level'].unique())}")
print(f"Job titles: {df_original['job_title'].nunique()}")
print(f"Countries: {df_original['company_location'].nunique()}")

print(f"\n{'=' * 60}")
print("DATASET B: Enriched Features (ai_jobs + tech)")
print("=" * 60)
print(f"Rows: {len(df_enriched):,}")
print(f"Columns: {len(df_enriched.columns)}")
print(f"Salary range: ${df_enriched['salary_in_usd'].min():,.0f} - ${df_enriched['salary_in_usd'].max():,.0f}")
print(f"Mean salary: ${df_enriched['salary_in_usd'].mean():,.0f}")
print(f"Experience levels: {sorted(df_enriched['experience_level'].unique())}")
print(f"Education levels: {sorted(df_enriched['education_level'].unique())}")
print(f"Work settings: {sorted(df_enriched['work_setting'].unique())}")
print(f"Industries: {df_enriched['industry'].nunique()}")
print(f"Job titles: {df_enriched['job_title'].nunique()}")
print(f"Countries: {df_enriched['company_location'].nunique()}")

DATASET A: Original Features (all sources)
Rows: 173,453
Columns: ['work_year', 'job_title', 'experience_level', 'employment_type', 'salary_in_usd', 'company_location', 'company_size']
Salary range: $5,132 - $811,600
Mean salary: $123,354
Experience levels: ['Entry', 'Executive', 'Lead', 'Mid', 'Senior']
Job titles: 442
Countries: 178

DATASET B: Enriched Features (ai_jobs + tech)
Rows: 102,000
Columns: 41
Salary range: $9,000 - $811,600
Mean salary: $103,135
Experience levels: ['Entry', 'Executive', 'Lead', 'Mid', 'Senior']
Education levels: ["Bachelor's", 'High School', "Master's", 'Other', 'PhD']
Work settings: ['Hybrid', 'Onsite', 'Remote']
Industries: 20
Job titles: 28
Countries: 21
